# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_attenuation
from functions import laser_set_standard, laser_get_standard
import snspd
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260415-36644-qcodes.log
Experiment loaded. Last ID no: 483


# Instruments

In [5]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

# Light Counts

Used parameters from 4-5 Counts vs Current.  

In [6]:
from functions import set_thresholds
v_scale_old_device = 50e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
threshold1, threshold2 = set_thresholds(yoko.current()) 
MS.write(f'SEARCH:SEARCH1:TRIGger:A:EDGE:THReshold {threshold1}')
MS.write(f'SEARCH:SEARCH2:TRIGger:A:EDGE:THReshold {threshold2}')

In [7]:
ID = params.att_info_id # use the ID from snspd to get attenuator range 
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator']

# set to maximum voltage and maximum attenuation
p_att.write(f'VOLT {max(v_attenuator)}')
time.sleep(10)

# Set standard oscilloscope parameters for counts
v_scale_old_device = 50e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
osc_check_standard(MS)

# Set laser parameters 
laser_set_standard(laser, 1550e-9, 7)
time.sleep(10)
laser_get_standard(laser)

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')

time.sleep(10)


if MS.channels[0].clipping(): 
    print('Error: Clipping')

snspd_counts_vs_attenuation(MS, dmm, yoko, p_att, device_name=params.device_2_name, n_captures=10, interval=1, current=2.1e-6, v_attenuator=v_attenuator, station=station)


############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.05
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0
Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm
Laser enable status: True
update station
Starting experimental run with id: 484. 
484
Current is 2.1e-06
Starting V=7
This acquisition will take 10s
13 36
Starting V=6.95
This acquisition will take 10s
13 36
Starting V=6.9
This acquisition will take 10s
13 36
Starting V=6.85
This acquisition will take 10s
13 37
Starting V=6.8
This acquisition will take 10s
13 37
Starting V=6.75
This acquisition will take 10s
13 37
Starting V=6.7
This acquisition will take 10s
13 37
Starting V=6.65
This acquisition will take 10s
13 38
Starting V=6.6
This acquisition will take 10s
13 38
Starting V=6.55
This acquisition will take 10s
13 38
Starting V=6.5
This acquisition will take 10s
13 38
Starting V=6.45
This acquisition will take 10s
13 39
Starting V=6.4
This acquisition will take 10s
13 39
Starting V=6.35
This acquisition wil

Run same measurement again with vertical scale set to 25mV/div

In [8]:
ID = params.att_info_id # use the ID from snspd to get attenuator range 
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator']

# set to maximum voltage and maximum attenuation
p_att.write(f'VOLT {max(v_attenuator)}')
time.sleep(10)

# Set standard oscilloscope parameters for counts
v_scale_old_device = 25e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
osc_check_standard(MS)

# Set laser parameters 
laser_set_standard(laser, 1550e-9, 7)
time.sleep(10)
laser_get_standard(laser)

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')

time.sleep(10)


if MS.channels[0].clipping(): 
    print('Error: Clipping')

snspd_counts_vs_attenuation(MS, dmm, yoko, p_att, device_name=params.device_2_name, n_captures=10, interval=1, current=2.1e-6, v_attenuator=v_attenuator, station=station)


############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.025
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0
Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm
Laser enable status: True
update station
Starting experimental run with id: 485. 
485
Current is 2.1e-06
Starting V=7
This acquisition will take 10s
13 56
Starting V=6.95
This acquisition will take 10s
13 56
Starting V=6.9
This acquisition will take 10s
13 57
Starting V=6.85
This acquisition will take 10s
13 57
Starting V=6.8
This acquisition will take 10s
13 57
Starting V=6.75
This acquisition will take 10s
13 57
Starting V=6.7
This acquisition will take 10s
13 58
Starting V=6.65
This acquisition will take 10s
13 58
Starting V=6.6
This acquisition will take 10s
13 58
Starting V=6.55
This acquisition will take 10s
13 58
Starting V=6.5
This acquisition will take 10s
13 59
Starting V=6.45
This acquisition will take 10s
13 59
Starting V=6.4
This acquisition will take 10s
13 59
Starting V=6.35
This acquisition wi

In [9]:
ID = params.att_info_id # use the ID from snspd to get attenuator range 
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator']

# set to maximum voltage and maximum attenuation
p_att.write(f'VOLT {max(v_attenuator)}')
time.sleep(10)

# Set standard oscilloscope parameters for counts
v_scale_old_device = 75e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
osc_check_standard(MS)

# Set laser parameters 
laser_set_standard(laser, 1550e-9, 7)
time.sleep(10)
laser_get_standard(laser)

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')

time.sleep(10)


if MS.channels[0].clipping(): 
    print('Error: Clipping')

snspd_counts_vs_attenuation(MS, dmm, yoko, p_att, device_name=params.device_2_name, n_captures=10, interval=1, current=2.1e-6, v_attenuator=v_attenuator, station=station)


############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.075
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0
Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm
Laser enable status: True
update station
Starting experimental run with id: 486. 
486
Current is 2.1e-06
Starting V=7
This acquisition will take 10s
14 16
Starting V=6.95
This acquisition will take 10s
14 17
Starting V=6.9
This acquisition will take 10s
14 17
Starting V=6.85
This acquisition will take 10s
14 17
Starting V=6.8
This acquisition will take 10s
14 17
Starting V=6.75
This acquisition will take 10s
14 18
Starting V=6.7
This acquisition will take 10s
14 18
Starting V=6.65
This acquisition will take 10s
14 18
Starting V=6.6
This acquisition will take 10s
14 18
Starting V=6.55
This acquisition will take 10s
14 19
Starting V=6.5
This acquisition will take 10s
14 19
Starting V=6.45
This acquisition will take 10s
14 19
Starting V=6.4
This acquisition will take 10s
14 19
Starting V=6.35
This acquisition wi

In [10]:
laser.enable()

False

The laser appears to have no effect

In [17]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True


In [16]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


In [13]:
from functions import set_thresholds
v_scale_old_device = 50e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
threshold1, threshold2 = set_thresholds(yoko.current()) 
MS.write(f'SEARCH:SEARCH1:TRIGger:A:EDGE:THReshold {threshold1}')
MS.write(f'SEARCH:SEARCH2:TRIGger:A:EDGE:THReshold {threshold2}')